# Qwen-27B Product-Law Boundary Calibration Audit

The preceding graded intervention rejected a stable linear product-law response, but its fully natural image control also failed to show a reliable linear dose response.

[实验设置与运行说明](../../../docs/experiment_notes/10_exploratory_composition_and_gating/1bc34b0675c039c8.md)


In [ ]:
# Run once. Every later code cell reports both cell and inner-loop progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    global _ACTIVE_CELL_PROGRESS
    specification = line.strip() or 'Cell progress'
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', specification)
    if match:
        current, total, label = int(match.group(1)), int(match.group(2)), match.group(3)
        initial = current - 1
    else:
        current, total, label, initial = 1, 1, specification, 0
    started = time.monotonic()
    bar = tqdm(
        total=total, initial=initial, desc=label, unit='cell',
        dynamic_ncols=True, leave=True,
    )
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()

    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s", refresh=True
            )

    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        cell_stage('running')
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set()
        worker.join(timeout=2.0)
        elapsed = time.monotonic() - started
        if succeeded:
            bar.n = current
            bar.set_postfix_str(f"complete | elapsed {elapsed:.0f}s", refresh=True)
        else:
            bar.set_postfix_str(f"failed | elapsed {elapsed:.0f}s", refresh=True)
        bar.close()
        _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1)
_SETUP_PROGRESS.close()
print('Cell progress helper ready: 12 tracked cells.')


## 1. Frozen evidence and physical calibration


In [ ]:
%%cell_progress 1/12 Locate the completed graded experiment
cell_stage('Mounting Drive and locating the VisualTorque project')
from pathlib import Path
import hashlib
import json
import math

try:
    from google.colab import drive
    drive.mount('/content/drive')
except (ImportError, ModuleNotFoundError):
    print('Not running in Colab; using local project candidates.')

PROJECT_CANDIDATES = (
    Path('/content/drive/MyDrive/VisualTorque'),
    Path('/content/drive/MyDrive/VisualTorque/VisualTorque'),
    Path.cwd(), Path.cwd().parent,
)
PROJECT_ROOT = next((
    path for path in PROJECT_CANDIDATES
    if (path / 'results' / 'qwen27b_graded_product_law_v1').is_dir()
), None)
assert PROJECT_ROOT is not None, (
    'Could not find results/qwen27b_graded_product_law_v1.'
)

SOURCE_ROOT = PROJECT_ROOT / 'results' / 'qwen27b_graded_product_law_v1'
SOURCE_ANALYSIS = SOURCE_ROOT / 'analysis'
SHARD_DIR = SOURCE_ROOT / 'family_shards'
OUTPUT_ROOT = (
    PROJECT_ROOT / 'results' / 'qwen27b_productlaw_boundary_calibration_v1'
)
ANALYSIS_DIR = OUTPUT_ROOT / 'analysis'
FIGURE_DIR = ANALYSIS_DIR / 'figures'
for path in (OUTPUT_ROOT, ANALYSIS_DIR, FIGURE_DIR):
    path.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return 'sha256:' + digest.hexdigest()

def read_jsonl(path):
    with Path(path).open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

def write_json_atomic(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(
        json.dumps(value, indent=2, ensure_ascii=False) + '\n',
        encoding='utf-8',
    )
    temporary.replace(path)

print('Project:', PROJECT_ROOT)
print('Source:', SOURCE_ROOT)
print('Output:', OUTPUT_ROOT)


In [ ]:
%%cell_progress 2/12 Lock and load the diagnostic evidence
cell_stage('Validating completion files, manifest, and all family shards')
from datetime import datetime, timezone
from collections import Counter, defaultdict
import numpy as np
import pandas as pd

RUN_PATH = SOURCE_ROOT / 'RUN_COMPLETE.json'
MANIFEST_PATH = SOURCE_ROOT / 'graded_product_manifest.jsonl'
MANIFEST_LOCK_PATH = SOURCE_ROOT / 'manifest_lock.json'
PROTOCOL_PATH = SOURCE_ROOT / 'frozen_protocol.json'
DECISION_PATH = SOURCE_ANALYSIS / 'graded_product_decision.json'
AUDIT_PATH = SOURCE_ANALYSIS / 'robustness_audit.json'
for required in (
    RUN_PATH, MANIFEST_PATH, MANIFEST_LOCK_PATH,
    PROTOCOL_PATH, DECISION_PATH, AUDIT_PATH,
):
    assert required.is_file(), required

source_run = json.loads(RUN_PATH.read_text(encoding='utf-8'))
source_decision = json.loads(DECISION_PATH.read_text(encoding='utf-8'))
source_audit = json.loads(AUDIT_PATH.read_text(encoding='utf-8'))
assert source_run['primary_decision'] == 'context_invariant_or_unreliable_factor_effect'
assert source_decision['primary_decision'] == source_run['primary_decision']
assert source_audit['physics_violations'] == 0
assert source_audit['nonfinite_count'] == 0

FACTORS = ('weight', 'distance')
BACKGROUNDS = ('b1', 'b2', 'b3')
CONDITIONS = (
    'additive_total', 'joint_restore', 'joint_random',
    'whole_donor', 'natural_hybrid', 'self_replay',
)
FAMILIES_PER_FACTOR = 12
BOOTSTRAP_REPLICATES = 10000
ALPHA = 0.05
SEED = 20261014
BEHAVIOR_ACCURACY_FLOOR = 2 / 3

diagnostic_identity = {
    'schema_version': 'productlaw-boundary-calibration-1.0',
    'status': 'analysis-only diagnostic; no model selection',
    'source_run_sha256': sha256_file(RUN_PATH),
    'source_manifest_sha256': sha256_file(MANIFEST_PATH),
    'source_protocol_sha256': sha256_file(PROTOCOL_PATH),
    'factors': list(FACTORS),
    'responses': ['natural_hybrid', 'additive_total'],
    'primary_comparison': 'boundary-aware versus raw-dose held-background error',
    'bootstrap_unit': 'factorial family',
    'bootstrap_replicates': BOOTSTRAP_REPLICATES,
    'seed': SEED,
}
LOCK_PATH = OUTPUT_ROOT / 'diagnostic_protocol.json'
if LOCK_PATH.is_file():
    previous = json.loads(LOCK_PATH.read_text(encoding='utf-8'))
    assert previous.get('identity') == diagnostic_identity, (
        'Existing diagnostic protocol differs. Use a new output directory.'
    )
else:
    write_json_atomic(LOCK_PATH, {
        'identity': diagnostic_identity,
        'locked_at_utc': datetime.now(timezone.utc).isoformat(),
    })

manifest = read_jsonl(MANIFEST_PATH)
manifest_by_family = {row['family_id']: row for row in manifest}
assert len(manifest) == 2 * FAMILIES_PER_FACTOR

records = []
shard_paths = sorted(SHARD_DIR.glob('*.json'))
assert len(shard_paths) == len(manifest)
for path in shard_paths:
    document = json.loads(path.read_text(encoding='utf-8'))
    records.extend(document['records'])
record_frame = pd.DataFrame(records)
assert len(record_frame) == len(manifest) * len(BACKGROUNDS) * len(CONDITIONS)
assert set(record_frame['condition']) == set(CONDITIONS)
print('Families / records:', len(manifest), len(record_frame))
print('Source decision:', source_run['primary_decision'])


In [ ]:
%%cell_progress 3/12 Reconstruct exact physical boundary coordinates
cell_stage('Joining target deficits, hybrid clearances, and model margins')

def opposite_side(side):
    return 'right' if side == 'left' else 'left'

def oriented_net(role, changed_side):
    outcome = str(role['outcome'])
    magnitude = abs(int(role['net_torque']))
    if outcome == changed_side:
        return float(magnitude)
    if outcome == 'balanced':
        return 0.0
    assert outcome == opposite_side(changed_side), (outcome, changed_side)
    return float(-magnitude)

def semantic_margin(record, positive, negative, clean=False):
    if clean:
        semantics = record['clean_candidate_semantics']
        logprobs = record['clean_candidate_logprobs']
    else:
        semantics = record['candidate_semantics']
        logprobs = record['candidate_logprobs']
    mapping = dict(zip(semantics, logprobs))
    return float(mapping[positive] - mapping[negative])

def gold_margin(record, gold, clean=False):
    if clean:
        semantics = list(record['clean_candidate_semantics'])
        logprobs = list(record['clean_candidate_logprobs'])
    else:
        semantics = list(record['candidate_semantics'])
        logprobs = list(record['candidate_logprobs'])
    mapping = dict(zip(semantics, logprobs))
    return float(mapping[gold] - max(
        value for semantic, value in mapping.items() if semantic != gold
    ))

cell_rows = []
for (factor, family_id, background), current in record_frame.groupby([
    'factor', 'family_id', 'background'
]):
    assert len(current) == len(CONDITIONS)
    by_condition = {
        row['condition']: row for _, row in current.iterrows()
    }
    family = manifest_by_family[family_id]
    changed_side = str(family['changed_side'])
    negative = opposite_side(changed_side)
    target_role_name = f'target_{background}'
    hybrid_role_name = f'hybrid_{background}'
    target_role = family['roles'][target_role_name]
    hybrid_role = family['roles'][hybrid_role_name]
    target_coordinate = oriented_net(target_role, changed_side)
    hybrid_coordinate = oriented_net(hybrid_role, changed_side)
    physical_dose = float(family['background_doses'][background])
    assert target_coordinate < 0
    assert abs((hybrid_coordinate - target_coordinate) - physical_dose) < 1e-9
    target_deficit = -target_coordinate
    normalized_dose = physical_dose / target_deficit
    if hybrid_coordinate < 0:
        transition = 'no_flip'
    elif hybrid_coordinate == 0:
        transition = 'balanced'
    else:
        transition = 'flip'

    natural = by_condition['natural_hybrid']
    additive = by_condition['additive_total']
    restored = by_condition['joint_restore']
    random = by_condition['joint_random']
    clean_prediction = str(additive['clean_predicted_semantic'])
    hybrid_gold = str(hybrid_role['outcome'])
    row = {
        'factor': factor,
        'family_id': family_id,
        'background': background,
        'changed_side': changed_side,
        'negative_side': negative,
        'transition': transition,
        'physical_dose': physical_dose,
        'target_deficit': target_deficit,
        'normalized_dose': normalized_dose,
        'target_oriented_net': target_coordinate,
        'hybrid_oriented_net': hybrid_coordinate,
        'clean_prediction': clean_prediction,
        'natural_prediction': str(natural['predicted_semantic']),
        'additive_prediction': str(additive['predicted_semantic']),
        'target_gold': str(target_role['outcome']),
        'hybrid_gold': hybrid_gold,
        'clean_correct': float(clean_prediction == str(target_role['outcome'])),
        'natural_correct': float(str(natural['predicted_semantic']) == hybrid_gold),
        'both_natural_endpoints_correct': float(
            clean_prediction == str(target_role['outcome'])
            and str(natural['predicted_semantic']) == hybrid_gold
        ),
        'additive_matches_hybrid_gold': float(
            str(additive['predicted_semantic']) == hybrid_gold
        ),
        'additive_matches_natural_prediction': float(
            str(additive['predicted_semantic']) == str(natural['predicted_semantic'])
        ),
        'clean_margin': semantic_margin(additive, changed_side, negative, clean=True),
        'natural_changed_margin': semantic_margin(natural, changed_side, negative),
        'additive_changed_margin': semantic_margin(additive, changed_side, negative),
        'natural_effect': float(natural['answer_effect']),
        'additive_effect': float(additive['answer_effect']),
        'restored_effect': float(restored['answer_effect']),
        'random_effect': float(random['answer_effect']),
        'route_attenuation': float(
            additive['answer_effect'] - restored['answer_effect']
        ),
        'route_specificity': float(
            random['answer_effect'] - restored['answer_effect']
        ),
        'clean_gold_margin': gold_margin(
            additive, str(target_role['outcome']), clean=True
        ),
        'natural_gold_margin': gold_margin(natural, hybrid_gold),
    }
    row['natural_positive_direction'] = float(row['natural_effect'] > 0)
    row['additive_positive_direction'] = float(row['additive_effect'] > 0)
    cell_rows.append(row)

calibration_frame = pd.DataFrame(cell_rows)
assert len(calibration_frame) == len(manifest) * len(BACKGROUNDS)
calibration_frame.to_csv(ANALYSIS_DIR / 'calibrated_cells.csv', index=False)
display(calibration_frame.groupby(['factor', 'transition'])[[
    'clean_correct', 'natural_correct', 'both_natural_endpoints_correct',
    'natural_effect', 'additive_effect',
]].agg(['count', 'mean']).round(4))


In [ ]:
%%cell_progress 4/12 Audit the fully natural behavioral positive control
cell_stage('Measuring accuracy, direction, monotonicity, and transition behavior')

def simple_slope(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    centered = x - x.mean()
    denominator = float(np.sum(centered ** 2))
    return float(np.sum(centered * (y - y.mean())) / denominator)

def spearman(x, y):
    x_rank = pd.Series(x).rank(method='average').to_numpy(dtype=float)
    y_rank = pd.Series(y).rank(method='average').to_numpy(dtype=float)
    if np.std(x_rank) == 0 or np.std(y_rank) == 0:
        return 0.0
    return float(np.corrcoef(x_rank, y_rank)[0, 1])

family_rows = []
for (factor, family_id), current in calibration_frame.groupby([
    'factor', 'family_id'
]):
    current = current.sort_values('physical_dose')
    natural_values = current['natural_effect'].to_numpy(dtype=float)
    additive_values = current['additive_effect'].to_numpy(dtype=float)
    row = {
        'factor': factor,
        'family_id': family_id,
        'clean_accuracy': float(current['clean_correct'].mean()),
        'natural_accuracy': float(current['natural_correct'].mean()),
        'both_accuracy': float(current['both_natural_endpoints_correct'].mean()),
        'natural_positive_direction_rate': float(
            current['natural_positive_direction'].mean()
        ),
        'additive_positive_direction_rate': float(
            current['additive_positive_direction'].mean()
        ),
        'additive_hybrid_match_rate': float(
            current['additive_matches_hybrid_gold'].mean()
        ),
        'additive_natural_match_rate': float(
            current['additive_matches_natural_prediction'].mean()
        ),
        'natural_raw_slope': simple_slope(
            current['physical_dose'], natural_values
        ),
        'additive_raw_slope': simple_slope(
            current['physical_dose'], additive_values
        ),
        'natural_boundary_slope': simple_slope(
            current['hybrid_oriented_net'], current['natural_changed_margin']
        ),
        'natural_raw_spearman': spearman(
            current['physical_dose'], natural_values
        ),
        'additive_raw_spearman': spearman(
            current['physical_dose'], additive_values
        ),
        'natural_monotonic': float(np.all(np.diff(natural_values) > 0)),
        'additive_monotonic': float(np.all(np.diff(additive_values) > 0)),
    }
    family_rows.append(row)
family_frame = pd.DataFrame(family_rows)
assert len(family_frame) == len(manifest)
family_frame.to_csv(ANALYSIS_DIR / 'family_behavior_metrics.csv', index=False)
display(family_frame.groupby('factor').mean(numeric_only=True).round(5))


## 2. Boundary-aware prediction and internal alignment


In [ ]:
%%cell_progress 5/12 Compare raw-dose and boundary-aware explanations
cell_stage('Running leave-one-background-out prediction within each family')

def ridge_fit_predict(train, test, features, family_levels, response, ridge=1e-6):
    family_to_index = {
        family: index for index, family in enumerate(family_levels)
    }
    def design(frame):
        one_hot = np.zeros((len(frame), len(family_levels)), dtype=float)
        for row_index, family in enumerate(frame['family_id'].astype(str)):
            one_hot[row_index, family_to_index[family]] = 1.0
        continuous = frame[features].to_numpy(dtype=float)
        return np.column_stack([one_hot, continuous])
    X_train, X_test = design(train), design(test)
    y_train = train[response].to_numpy(dtype=float)
    penalty = ridge * np.eye(X_train.shape[1])
    coefficients = np.linalg.solve(
        X_train.T @ X_train + penalty, X_train.T @ y_train
    )
    return X_test @ coefficients

MODEL_FEATURES = {
    'context_free': ['clean_margin'],
    'raw_dose': ['clean_margin', 'physical_dose'],
    'boundary_clearance': ['clean_margin', 'hybrid_oriented_net'],
    'dose_plus_deficit': ['clean_margin', 'physical_dose', 'target_deficit'],
    'normalized_boundary': ['clean_margin', 'normalized_dose'],
}
RESPONSES = ('natural_effect', 'additive_effect')
prediction_rows = []
for factor in FACTORS:
    current = calibration_frame[calibration_frame['factor'] == factor]
    family_levels = sorted(current['family_id'].astype(str).unique())
    for held_background in BACKGROUNDS:
        train = current[current['background'] != held_background]
        test = current[current['background'] == held_background].copy()
        for response in RESPONSES:
            truth = test[response].to_numpy(dtype=float)
            predictions = {
                name: ridge_fit_predict(
                    train, test, features, family_levels, response
                )
                for name, features in MODEL_FEATURES.items()
            }
            for row_index, (_, row) in enumerate(test.iterrows()):
                output = {
                    'factor': factor,
                    'family_id': row['family_id'],
                    'background': held_background,
                    'response': response,
                    'truth': float(truth[row_index]),
                }
                for name, values in predictions.items():
                    output[f'{name}_prediction'] = float(values[row_index])
                    output[f'{name}_squared_error'] = float(
                        (truth[row_index] - values[row_index]) ** 2
                    )
                prediction_rows.append(output)

prediction_frame = pd.DataFrame(prediction_rows)
prediction_frame.to_csv(
    ANALYSIS_DIR / 'held_background_calibration_predictions.csv', index=False
)
error_columns = [f'{name}_squared_error' for name in MODEL_FEATURES]
family_prediction = prediction_frame.groupby([
    'factor', 'family_id', 'response'
], as_index=False)[error_columns].mean()
for comparator in (
    'context_free', 'raw_dose', 'normalized_boundary'
):
    family_prediction[f'boundary_vs_{comparator}'] = (
        family_prediction[f'{comparator}_squared_error']
        - family_prediction['boundary_clearance_squared_error']
    )
family_prediction['dose_deficit_vs_raw_dose'] = (
    family_prediction['raw_dose_squared_error']
    - family_prediction['dose_plus_deficit_squared_error']
)
family_prediction.to_csv(
    ANALYSIS_DIR / 'family_boundary_prediction_metrics.csv', index=False
)
display(family_prediction.groupby(['factor', 'response']).mean(
    numeric_only=True
).round(6))


In [ ]:
%%cell_progress 6/12 Test whether internal changes track natural changes
cell_stage('Computing family-centered alignment and transition agreement')

alignment_rows = []
for (factor, family_id), current in calibration_frame.groupby([
    'factor', 'family_id'
]):
    natural = current['natural_effect'].to_numpy(dtype=float)
    additive = current['additive_effect'].to_numpy(dtype=float)
    natural_centered = natural - natural.mean()
    additive_centered = additive - additive.mean()
    denominator = float(np.sum(natural_centered ** 2))
    alignment_slope = (
        float(np.sum(natural_centered * additive_centered) / denominator)
        if denominator > 0 else 0.0
    )
    alignment_rows.append({
        'factor': factor,
        'family_id': family_id,
        'alignment_slope': alignment_slope,
        'direction_agreement': float(np.mean(
            np.sign(natural) == np.sign(additive)
        )),
        'positive_joint_direction': float(np.mean(
            (natural > 0) & (additive > 0)
        )),
        'prediction_agreement': float(
            current['additive_matches_natural_prediction'].mean()
        ),
        'hybrid_gold_agreement': float(
            current['additive_matches_hybrid_gold'].mean()
        ),
        'route_attenuation_mean': float(current['route_attenuation'].mean()),
        'route_specificity_mean': float(current['route_specificity'].mean()),
    })
alignment_frame = pd.DataFrame(alignment_rows)
alignment_frame.to_csv(ANALYSIS_DIR / 'family_internal_natural_alignment.csv', index=False)
display(alignment_frame.groupby('factor').mean(numeric_only=True).round(5))


In [ ]:
%%cell_progress 7/12 Bootstrap family-level diagnostic intervals
cell_stage('Estimating uncertainty without treating backgrounds as independent')

def stable_seed(key):
    digest = hashlib.sha256(f'{SEED}:{key}'.encode('utf-8')).hexdigest()
    return int(digest[:16], 16) % (2**63 - 1)

def bootstrap_mean(values, key):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    assert len(values) >= 8, (key, len(values))
    generator = np.random.default_rng(stable_seed(key))
    draws = generator.integers(
        0, len(values), size=(BOOTSTRAP_REPLICATES, len(values))
    )
    estimates = values[draws].mean(axis=1)
    return {
        'estimate': float(values.mean()),
        'ci_low': float(np.quantile(estimates, ALPHA / 2)),
        'ci_high': float(np.quantile(estimates, 1 - ALPHA / 2)),
        'n_families': int(len(values)),
    }

test_rows = []
for factor in FACTORS:
    behavior = family_frame[family_frame['factor'] == factor]
    alignment = alignment_frame[alignment_frame['factor'] == factor]
    for metric in (
        'clean_accuracy', 'natural_accuracy', 'both_accuracy',
        'natural_positive_direction_rate', 'additive_positive_direction_rate',
        'natural_raw_slope', 'additive_raw_slope',
        'natural_raw_spearman', 'additive_raw_spearman',
        'natural_monotonic', 'additive_monotonic',
    ):
        summary = bootstrap_mean(behavior[metric], f'{factor}:behavior:{metric}')
        test_rows.append({
            'factor': factor, 'family': 'behavior', 'metric': metric, **summary
        })
    for metric in (
        'alignment_slope', 'direction_agreement', 'positive_joint_direction',
        'prediction_agreement', 'hybrid_gold_agreement',
        'route_attenuation_mean', 'route_specificity_mean',
    ):
        summary = bootstrap_mean(alignment[metric], f'{factor}:alignment:{metric}')
        test_rows.append({
            'factor': factor, 'family': 'alignment', 'metric': metric, **summary
        })
    for response in RESPONSES:
        prediction = family_prediction[
            (family_prediction['factor'] == factor)
            & (family_prediction['response'] == response)
        ]
        for metric in (
            'boundary_vs_context_free', 'boundary_vs_raw_dose',
            'boundary_vs_normalized_boundary', 'dose_deficit_vs_raw_dose',
        ):
            summary = bootstrap_mean(
                prediction[metric], f'{factor}:{response}:{metric}'
            )
            test_rows.append({
                'factor': factor,
                'family': f'prediction:{response}',
                'metric': metric,
                **summary,
            })

test_frame = pd.DataFrame(test_rows)
test_frame.to_csv(ANALYSIS_DIR / 'calibration_tests.csv', index=False)
display(test_frame.round(6))


In [ ]:
%%cell_progress 8/12 Separate boundary regimes and competent endpoints
cell_stage('Reporting all data and descriptive competence strata')

strata_rows = []
for keys, current in calibration_frame.groupby(['factor', 'transition']):
    factor, transition = keys
    for stratum, subset in (
        ('all', current),
        ('both_natural_endpoints_correct', current[
            current['both_natural_endpoints_correct'] == 1
        ]),
    ):
        if len(subset) == 0:
            continue
        strata_rows.append({
            'factor': factor,
            'transition': transition,
            'stratum': stratum,
            'n_cells': len(subset),
            'n_families': subset['family_id'].nunique(),
            'clean_accuracy': subset['clean_correct'].mean(),
            'natural_accuracy': subset['natural_correct'].mean(),
            'natural_effect': subset['natural_effect'].mean(),
            'additive_effect': subset['additive_effect'].mean(),
            'route_attenuation': subset['route_attenuation'].mean(),
            'route_specificity': subset['route_specificity'].mean(),
            'additive_hybrid_match': subset['additive_matches_hybrid_gold'].mean(),
        })
strata_frame = pd.DataFrame(strata_rows)
strata_frame.to_csv(ANALYSIS_DIR / 'transition_competence_strata.csv', index=False)
display(strata_frame.round(5))

# Precision projections describe within-model statistical precision only.
projection_rows = []
for factor in FACTORS:
    for metric in ('natural_raw_slope', 'additive_raw_slope'):
        summary = test_frame[
            (test_frame['factor'] == factor)
            & (test_frame['metric'] == metric)
        ].iloc[0]
        current_halfwidth = (
            float(summary['ci_high']) - float(summary['ci_low'])
        ) / 2
        for families in (12, 24, 32, 40, 64):
            projection_rows.append({
                'factor': factor,
                'metric': metric,
                'families': families,
                'projected_ci_halfwidth': current_halfwidth * math.sqrt(
                    FAMILIES_PER_FACTOR / families
                ),
                'scope': 'within Qwen-27B only; not cross-model generalization',
            })
projection_frame = pd.DataFrame(projection_rows)
projection_frame.to_csv(ANALYSIS_DIR / 'precision_projection.csv', index=False)
display(projection_frame[projection_frame['families'].isin([12, 24, 40])].round(6))


## 3. Diagnostic decision and outputs


In [ ]:
%%cell_progress 9/12 Derive diagnostic outcomes and next experiment
cell_stage('Classifying measurement failure without revising prior claims')

def result(factor, family, metric):
    return test_frame[
        (test_frame['factor'] == factor)
        & (test_frame['family'] == family)
        & (test_frame['metric'] == metric)
    ].iloc[0]

factor_decisions = {}
for factor in FACTORS:
    natural_accuracy = result(factor, 'behavior', 'natural_accuracy')
    natural_direction = result(
        factor, 'behavior', 'natural_positive_direction_rate'
    )
    boundary_natural = result(
        factor, 'prediction:natural_effect', 'boundary_vs_raw_dose'
    )
    boundary_additive = result(
        factor, 'prediction:additive_effect', 'boundary_vs_raw_dose'
    )
    alignment = result(factor, 'alignment', 'alignment_slope')
    direction_agreement = result(
        factor, 'alignment', 'direction_agreement'
    )
    gates = {
        'natural_endpoint_accuracy_adequate': bool(
            natural_accuracy['estimate'] >= BEHAVIOR_ACCURACY_FLOOR
            and natural_accuracy['ci_low'] > 1 / 3
        ),
        'natural_change_has_positive_direction': bool(
            natural_direction['ci_low'] > 0.5
        ),
        'boundary_beats_raw_dose_for_natural_images': bool(
            boundary_natural['ci_low'] > 0
        ),
        'boundary_beats_raw_dose_for_internal_intervention': bool(
            boundary_additive['ci_low'] > 0
        ),
        'internal_effect_tracks_natural_magnitude': bool(
            alignment['ci_low'] > 0
        ),
        'internal_effect_tracks_natural_direction': bool(
            direction_agreement['ci_low'] > 0.5
        ),
    }
    natural_reliable = (
        gates['natural_endpoint_accuracy_adequate']
        and gates['natural_change_has_positive_direction']
    )
    boundary_candidate = gates['boundary_beats_raw_dose_for_natural_images']
    internal_tracks = (
        gates['internal_effect_tracks_natural_magnitude']
        and gates['internal_effect_tracks_natural_direction']
    )
    if not natural_reliable:
        diagnostic = 'natural_behavior_positive_control_unreliable'
        next_test = 'competence-stratified matched-clearance confirmation'
    elif boundary_candidate and internal_tracks:
        diagnostic = 'boundary_gated_internal_recombination_candidate'
        next_test = 'independent matched-clearance graded confirmation'
    elif boundary_candidate:
        diagnostic = 'boundary_metric_explains_natural_but_not_internal_effect'
        next_test = 'matched-clearance internal intervention with frozen route'
    elif internal_tracks:
        diagnostic = 'internal_tracks_natural_without_boundary_advantage'
        next_test = 'nonlinear transition-focused confirmation'
    else:
        diagnostic = 'stable_composition_remains_unsupported'
        next_test = 'targeted shortcut-error rescue and induction'
    factor_decisions[factor] = {
        'diagnostic': diagnostic,
        'recommended_next_test': next_test,
        'gates': gates,
    }

diagnostics = {value['diagnostic'] for value in factor_decisions.values()}
if diagnostics == {'natural_behavior_positive_control_unreliable'}:
    PRIMARY_DIAGNOSTIC = 'natural_positive_control_limits_both_factor_tests'
elif any(
    'boundary' in value['diagnostic']
    for value in factor_decisions.values()
):
    PRIMARY_DIAGNOSTIC = 'boundary_calibration_requires_independent_confirmation'
elif any(
    'tracks_natural' in value['diagnostic']
    for value in factor_decisions.values()
):
    PRIMARY_DIAGNOSTIC = 'nonlinear_natural_tracking_requires_confirmation'
else:
    PRIMARY_DIAGNOSTIC = 'graded_composition_remains_unsupported_after_calibration'

decision_document = {
    'schema_version': 'productlaw-boundary-calibration-decision-1.0',
    'primary_diagnostic': PRIMARY_DIAGNOSTIC,
    'source_confirmatory_decision_unchanged': source_run['primary_decision'],
    'factor_decisions': factor_decisions,
    'claim_boundary': (
        'This analysis diagnoses measurement and competence; it neither selects '
        'a new neural route nor reverses the frozen graded confirmation.'
    ),
}
write_json_atomic(ANALYSIS_DIR / 'calibration_decision.json', decision_document)
print('Primary diagnostic:', PRIMARY_DIAGNOSTIC)
display(pd.DataFrame([
    {'factor': factor, **value['gates'],
     'diagnostic': value['diagnostic'],
     'next_test': value['recommended_next_test']}
    for factor, value in factor_decisions.items()
]))


In [ ]:
%%cell_progress 10/12 Audit reconstruction and diagnostic independence
cell_stage('Checking coordinates, counts, support, and frozen provenance')

coordinate_errors = []
for _, row in calibration_frame.iterrows():
    if abs(
        row['hybrid_oriented_net'] - row['target_oriented_net']
        - row['physical_dose']
    ) > 1e-9:
        coordinate_errors.append(row['family_id'])
assert not coordinate_errors
assert len(calibration_frame) == 72
assert calibration_frame.groupby('factor')['family_id'].nunique().to_dict() == {
    factor: FAMILIES_PER_FACTOR for factor in FACTORS
}
assert calibration_frame.groupby([
    'factor', 'family_id'
]).size().eq(len(BACKGROUNDS)).all()
assert set(calibration_frame['transition']) <= {'no_flip', 'balanced', 'flip'}
assert source_audit['selection_uses_model_outputs'] is False
assert source_audit['route_selected_from_current_data'] is False

audit_document = {
    'schema_version': 'productlaw-boundary-calibration-audit-1.0',
    'source_records': len(record_frame),
    'calibrated_cells': len(calibration_frame),
    'families': len(manifest),
    'coordinate_errors': len(coordinate_errors),
    'source_physics_violations': source_audit['physics_violations'],
    'source_nonfinite_count': source_audit['nonfinite_count'],
    'source_global_scene_disjoint': source_audit['global_scene_disjoint'],
    'source_immediate_prior_overlap': (
        source_audit['immediate_prior_scene_overlap_count']
    ),
    'selection_uses_model_outputs': False,
    'selects_new_route': False,
    'changes_source_confirmatory_decision': False,
    'diagnostic_protocol_sha256': sha256_file(LOCK_PATH),
}
write_json_atomic(ANALYSIS_DIR / 'calibration_audit.json', audit_document)
display(pd.DataFrame([audit_document]).T.rename(columns={0: 'value'}))


In [ ]:
%%cell_progress 11/12 Save the paper-facing calibration figure
cell_stage('Rendering behavior, boundary prediction, alignment, and precision')
import matplotlib.pyplot as plt

colors = {'weight': '#D97706', 'distance': '#2563EB'}
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

behavior_metrics = ('clean_accuracy', 'natural_accuracy', 'both_accuracy')
x = np.arange(len(behavior_metrics))
width = 0.35
for index, factor in enumerate(FACTORS):
    values = [
        result(factor, 'behavior', metric)['estimate']
        for metric in behavior_metrics
    ]
    axes[0, 0].bar(
        x + (index - 0.5) * width, values, width,
        label=factor.title(), color=colors[factor],
    )
axes[0, 0].axhline(1 / 3, color='black', linestyle='--', linewidth=1)
axes[0, 0].set_xticks(x, ['Target', 'Natural hybrid', 'Both'])
axes[0, 0].set_ylim(0, 1)
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].set_title('A. Natural-image positive control')
axes[0, 0].legend(frameon=False)

for factor in FACTORS:
    current = calibration_frame[calibration_frame['factor'] == factor]
    axes[0, 1].scatter(
        current['hybrid_oriented_net'], current['natural_effect'],
        alpha=0.65, s=36, color=colors[factor], label=factor.title(),
    )
axes[0, 1].axvline(0, color='black', linewidth=1)
axes[0, 1].axhline(0, color='black', linewidth=1)
axes[0, 1].set_xlabel('Hybrid net torque toward changed side')
axes[0, 1].set_ylabel('Natural changed-side margin shift')
axes[0, 1].set_title('B. Physical boundary calibration')
axes[0, 1].legend(frameon=False)

prediction_metrics = (
    'boundary_vs_raw_dose', 'dose_deficit_vs_raw_dose'
)
labels = ('Boundary clearance', 'Dose + target deficit')
x = np.arange(len(prediction_metrics))
for index, factor in enumerate(FACTORS):
    current = family_prediction[
        (family_prediction['factor'] == factor)
        & (family_prediction['response'] == 'natural_effect')
    ]
    axes[1, 0].bar(
        x + (index - 0.5) * width,
        [current[metric].mean() for metric in prediction_metrics],
        width, color=colors[factor], label=factor.title(),
    )
axes[1, 0].axhline(0, color='black', linewidth=1)
axes[1, 0].set_xticks(x, labels, rotation=12)
axes[1, 0].set_ylabel('MSE improvement over raw dose')
axes[1, 0].set_title('C. Held-background prediction')

for factor in FACTORS:
    current = projection_frame[
        (projection_frame['factor'] == factor)
        & (projection_frame['metric'] == 'additive_raw_slope')
    ]
    axes[1, 1].plot(
        current['families'], current['projected_ci_halfwidth'],
        marker='o', color=colors[factor], label=factor.title(),
    )
axes[1, 1].set_xlabel('Independent families per factor')
axes[1, 1].set_ylabel('Projected 95% CI half-width')
axes[1, 1].set_title('D. Within-model precision only')
axes[1, 1].legend(frameon=False)

fig.suptitle('Qwen-27B graded product-law calibration audit', fontsize=16)
fig.tight_layout()
figure_path = FIGURE_DIR / 'productlaw_boundary_calibration.png'
fig.savefig(figure_path, dpi=220, bbox_inches='tight')
plt.show()
print('Saved:', figure_path)


In [ ]:
%%cell_progress 12/12 Write the calibration and sample-size report
cell_stage('Saving reusable tables, interpretation, and completion metadata')

def interval(factor, family, metric):
    row = result(factor, family, metric)
    return f"{row['estimate']:.5f} [{row['ci_low']:.5f}, {row['ci_high']:.5f}]"

lines = [
    '# Product-Law Boundary Calibration Audit', '',
    f"**Primary diagnostic:** `{PRIMARY_DIAGNOSTIC}`", '',
    f"**Frozen source decision remains:** `{source_run['primary_decision']}`", '',
    '## Why this audit was required', '',
    (
        'The raw-dose confirmation failed, but its natural-image positive '
        'control also lacked a reliable linear slope. This audit separates '
        'unreliable natural behavior from a missing decision-boundary covariate '
        'and from internal-natural misalignment.'
    ), '',
]
for factor in FACTORS:
    decision = factor_decisions[factor]
    lines.extend([
        f"## {factor.title()}", '',
        f"- Diagnostic: `{decision['diagnostic']}`",
        f"- Recommended next test: {decision['recommended_next_test']}",
        f"- Natural hybrid accuracy: {interval(factor, 'behavior', 'natural_accuracy')}",
        f"- Natural positive-direction rate: {interval(factor, 'behavior', 'natural_positive_direction_rate')}",
        f"- Additive positive-direction rate: {interval(factor, 'behavior', 'additive_positive_direction_rate')}",
        f"- Boundary vs raw dose, natural: {interval(factor, 'prediction:natural_effect', 'boundary_vs_raw_dose')}",
        f"- Boundary vs raw dose, internal: {interval(factor, 'prediction:additive_effect', 'boundary_vs_raw_dose')}",
        f"- Internal-to-natural alignment slope: {interval(factor, 'alignment', 'alignment_slope')}",
        f"- Direction agreement: {interval(factor, 'alignment', 'direction_agreement')}",
        f"- Gates: `{json.dumps(decision['gates'], sort_keys=True)}`", '',
    ])
lines.extend([
    '## Sample-size interpretation', '',
    (
        'Twelve independent families per factor can detect large effects but '
        'leave wide intervals for small slopes. Increasing to 24–40 families '
        'improves within-Qwen-27B precision. It does not establish cross-model '
        'generalization; that requires a frozen replication on other models.'
    ), '',
    '## Claim boundary', '',
    (
        'This is a diagnostic reanalysis of the same support. Any newly '
        'suggested boundary-aware or competence-stratified result must be '
        'confirmed on an independent manifest before entering the main claim.'
    ), '',
])
report_path = ANALYSIS_DIR / 'PRODUCTLAW_BOUNDARY_CALIBRATION_REPORT.md'
report_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')

completion = {
    'schema_version': 'productlaw-boundary-calibration-complete-1.0',
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
    'primary_diagnostic': PRIMARY_DIAGNOSTIC,
    'source_decision_unchanged': source_run['primary_decision'],
    'factor_decisions': factor_decisions,
    'source_records': len(record_frame),
    'calibrated_cells': len(calibration_frame),
    'report_path': str(report_path),
    'figure_path': str(figure_path),
    'diagnostic_protocol_sha256': sha256_file(LOCK_PATH),
}
write_json_atomic(OUTPUT_ROOT / 'RUN_COMPLETE.json', completion)
print('Primary diagnostic:', PRIMARY_DIAGNOSTIC)
print('Report:', report_path)
print('Run complete:', OUTPUT_ROOT / 'RUN_COMPLETE.json')
